# 🏪 Sistema de Conteo de Clientes en Tienda mediante Visión por Computadora

Este notebook implementa un pipeline completo y autocontenido para la detección, seguimiento y conteo de clientes que acuden a comprar a un quiosco/tienda comercial.

### 📌 Métodos Implementados:
1. **Método 1: Visión por Computadora Tradicional (Sustracción de Fondo)**
   - Ajuste de brillo para compensar la baja iluminación del quiosco.
   - Enfoque y recorte vertical de la zona media-baja (eliminando cielo, edificios altos y pasto).
   - Cálculo del fondo estático por mediana en segundo de tienda vacía (`SEGUNDO_VACIO = 848`).
   - Detección de presencia, temporizador de permanencia (`SEG_MIN = 8s`) y conteo por zonas.
2. **Método 2: Deep Learning con YOLOv11 (Tracking de Personas)**
   - Tracking persistente de personas (`classes=[0]`) con YOLOv11n.
   - Detección de posición de los pies de cada cliente en las zonas de compra.
   - Asignación de ID único para evitar conteos duplicados.
3. **Módulo de Estabilización de Video (Opcional)**
   - Algoritmo de flujo óptico Lucas-Kanade y transformación afín para eliminar vibraciones de cámara.

---
> 💡 **Nota para Google Colab:** Si vas a ejecutar el método con YOLO, se recomienda activar GPU:  
> `Entorno de ejecución` ➔ `Cambiar tipo de entorno de ejecución` ➔ Seleccionar **T4 GPU**.


In [ ]:
# ==============================================================================
# 0. INSTALACIÓN E IMPORTACIÓN DE DEPENDENCIAS
# ==============================================================================
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Detectado entorno: Google Colab")
    print("📦 Instalando paquetes necesarios...")
    !pip install -q ultralytics opencv-python-headless matplotlib tqdm
except ImportError:
    IN_COLAB = False
    print("💻 Detectado entorno: Jupyter Local")

import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import HTML, display, Video
from base64 import b64encode

print("✅ Todas las librerías se importaron correctamente.")


## 📁 1. Configuración de Archivos y Zonas de Compra (ROIs)

Aquí se verifica el video de entrada y se configuran las 3 zonas del mostrador de la tienda.
- El notebook empaqueta por defecto las coordenadas calibradas de las 3 zonas:
  - **Zona 1**: Mostrador izquierdo (`x=151, y=511, w=39, h=150`)
  - **Zona 2**: Mostrador central (`x=198, y=511, w=34, h=152`)
  - **Zona 3**: Vitrina derecha (`x=240, y=510, w=49, h=151`)
- Si no existe `espacios.pkl`, se generará automáticamente con estas coordenadas.


In [ ]:
# ==============================================================================
# 1. VERIFICACIÓN DE VIDEO Y ZONAS
# ==============================================================================

# Prioriza el video estabilizado si existe; si no, usa el video original
if os.path.exists('estabilizado.mp4'):
    VIDEO_PATH = 'estabilizado.mp4'
elif os.path.exists('video.mp4'):
    VIDEO_PATH = 'video.mp4'
else:
    VIDEO_PATH = 'estabilizado.mp4'

if not os.path.exists(VIDEO_PATH):
    print(f"⚠️ No se encontró '{VIDEO_PATH}' en el directorio actual.")
    if IN_COLAB:
        from google.colab import files
        print("📥 Por favor, sube tu video (estabilizado.mp4 o video.mp4):")
        subidos = files.upload()
        if subidos:
            VIDEO_PATH = list(subidos.keys())[0]
            print(f"✅ Video subido: {VIDEO_PATH}")
    else:
        print("Asegúrate de colocar 'estabilizado.mp4' o 'video.mp4' en la misma carpeta del notebook.")
else:
    print(f"✅ Video encontrado: '{VIDEO_PATH}'")

# Coordenadas calibradas de las 3 zonas de mostrador (x, y, w, h) para ESCALA=0.7
ZONAS_DEFAULT = [
    (151, 511, 39, 150),  # Zona 1: Mostrador izquierdo
    (198, 511, 34, 152),  # Zona 2: Mostrador central
    (240, 510, 49, 151)   # Zona 3: Vitrina derecha
]

# Guardar o cargar espacios.pkl automáticamente
PKL_PATH = 'espacios.pkl'
if not os.path.exists(PKL_PATH):
    with open(PKL_PATH, 'wb') as f:
        pickle.dump(ZONAS_DEFAULT, f)
    print(f"📦 Se generó '{PKL_PATH}' automáticamente con las 3 zonas de la tienda.")
else:
    with open(PKL_PATH, 'rb') as f:
        ZONAS_DEFAULT = pickle.load(f)
    print(f"📦 '{PKL_PATH}' cargado exitosamente ({len(ZONAS_DEFAULT)} zonas).")

print(f"📍 Zonas configuradas: {ZONAS_DEFAULT}")


## 📐 2. Visualización de la Tienda y Zonas de Interés

A continuación visualizamos el fotograma de la tienda en el segundo donde no hay clientes (`SEGUNDO_VACIO = 848`), mostrando:
1. **Fotograma Completo**: Con las líneas horizontales de recorte donde se eliminarán el cielo y el pasto.
2. **Fotograma Enfocado**: Con brillo aumentado (`BRILLO = 40`) y recorte vertical enfocado exclusivamente en el quiosco.


In [ ]:
# ==============================================================================
# 2. VISUALIZACIÓN DE FOTOGRAMA Y RECORTE
# ==============================================================================
ESCALA = 0.7
SEGUNDO_VACIO = 848
BRILLO = 40
CORTE_SUPERIOR = 0.45   # Eliminar el 45% superior (cielo y pisos superiores)
CORTE_INFERIOR = 0.85   # Eliminar el pasto inferior

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_MSEC, SEGUNDO_VACIO * 1000)
ok, frame = cap.read()
cap.release()

if ok:
    frame = cv2.resize(frame, None, fx=ESCALA, fy=ESCALA)
    alto, ancho = frame.shape[:2]
    y_inicio = int(alto * CORTE_SUPERIOR)
    y_fin = int(alto * CORTE_INFERIOR)
    
    # Aplicar aumento de brillo
    frame_brillante = cv2.convertScaleAbs(frame, beta=BRILLO)
    
    # Recorte enfocado en la tienda
    frame_recortado = frame_brillante[y_inicio:y_fin, :].copy()
    
    # Coordenadas relativas al recorte (y - y_inicio)
    zonas_recortadas = [(x, y - y_inicio, w, h) for x, y, w, h in ZONAS_DEFAULT]
    
    # Dibujar zonas en el fotograma completo
    frame_full_draw = frame.copy()
    for i, (x, y, w, h) in enumerate(ZONAS_DEFAULT):
        cv2.rectangle(frame_full_draw, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame_full_draw, f"Z{i+1}", (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
    # Dibujar zonas en el fotograma recortado
    for i, (x, y, w, h) in enumerate(zonas_recortadas):
        cv2.rectangle(frame_recortado, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame_recortado, f"Zona {i+1}", (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Graficar lado a lado
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(frame_full_draw, cv2.COLOR_BGR2RGB))
    plt.axhline(y_inicio, color='red', linestyle='--', linewidth=2, label=f'Corte Superior ({CORTE_SUPERIOR*100:.0f}%)')
    plt.axhline(y_fin, color='blue', linestyle='--', linewidth=2, label=f'Corte Inferior ({CORTE_INFERIOR*100:.0f}%)')
    plt.title("Fotograma Original (Con Líneas de Recorte)", fontsize=13)
    plt.legend(loc='upper right')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(frame_recortado, cv2.COLOR_BGR2RGB))
    plt.title("Enfoque en Quiosco (Con Brillo y Zonas Alineadas)", fontsize=13)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No se pudo leer el fotograma del video. Verifica la ruta de VIDEO_PATH.")


## 🔄 3. (Opcional) Estabilización de Video por Flujo Óptico
Si tu video original fue grabado con pulso a mano o con viento, el movimiento de la cámara genera ruido en la sustracción de fondo.

Esta función toma `video.mp4` y genera `estabilizado.mp4` compensando los movimientos con respecto al fotograma de referencia.


In [ ]:
# ==============================================================================
# 3. FUNCIÓN DE ESTABILIZACIÓN (OPCIONAL)
# ==============================================================================
def estabilizar_video(video_origen='video.mp4', video_destino='estabilizado.mp4', segundo_vacio=848):
    if not os.path.exists(video_origen):
        print(f"⚠️ No existe {video_origen} para estabilizar.")
        return
        
    cap = cv2.VideoCapture(video_origen)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Fotograma de referencia (tienda vacía)
    cap.set(cv2.CAP_PROP_POS_MSEC, segundo_vacio * 1000)
    ok, ref = cap.read()
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    if not ok:
        print("❌ Error al leer fotograma de referencia.")
        cap.release()
        return
        
    alto, ancho = ref.shape[:2]
    out = cv2.VideoWriter(video_destino, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))
    
    ref_gris = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY)
    puntos_ref = cv2.goodFeaturesToTrack(ref_gris, maxCorners=400, qualityLevel=0.01, minDistance=20)
    
    print(f"🎬 Iniciando estabilización de {total} fotogramas...")
    pbar = tqdm(total=total, desc="Estabilizando")
    
    while True:
        ok, frame = cap.read()
        if not ok:
            break
            
        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        puntos, estado, _ = cv2.calcOpticalFlowPyrLK(ref_gris, gris, puntos_ref, None, winSize=(21, 21), maxLevel=4)
        buenos_ref = puntos_ref[estado == 1]
        buenos_act = puntos[estado == 1]
        
        M, _ = cv2.estimateAffinePartial2D(buenos_act, buenos_ref)
        if M is not None:
            frame = cv2.warpAffine(frame, M, (ancho, alto))
            
        out.write(frame)
        pbar.update(1)
        
    pbar.close()
    cap.release()
    out.release()
    print(f"✅ Video estabilizado generado en: {video_destino}")

# Descomenta la siguiente línea si deseas ejecutar la estabilización:
# estabilizar_video('video.mp4', 'estabilizado.mp4')


## 👥 4. Método 1: Visión por Computadora Tradicional (Sustracción de Fondo)

Este método funciona procesando el video de la siguiente manera:
1. **Fondo Mediana**: Se extrae una muestra de 15 fotogramas alrededor del segundo 848 (tienda vacía) y se calcula la mediana por píxel para eliminar cualquier objeto transitorio.
2. **Preprocesamiento en Vivo**: Cada fotograma recibe aumento de brillo (`BRILLO = 40`) y el recorte de la zona media-baja.
3. **Diferencia Absoluta**: `absdiff(fotograma, fondo)` -> `threshold` -> `medianBlur` -> `dilate`.
4. **Criterio de Conteo**:
   - Se calcula el porcentaje de píxeles activos en cada zona.
   - Si `porcentaje > UMBRAL (0.10)` durante `SEG_MIN (8s)` continuos, se registra como cliente atendido.
   - Al liberarse la zona durante `SEG_LIBRE (1s)`, queda lista para el siguiente cliente.
5. **Video de Salida**: Se genera `resultado_tradicional.mp4` con los recuadros coloreados (verde = libre, amarillo = ocupado, rojo = cliente confirmado).


In [ ]:
# ==============================================================================
# 4. EJECUCIÓN MÉTODO TRADICIONAL
# ==============================================================================
def procesar_metodo_tradicional(
    video_path,
    output_path='resultado_tradicional.mp4',
    zonas=ZONAS_DEFAULT,
    escala=0.7,
    brillo=40,
    corte_superior=0.45,
    corte_inferior=0.85,
    segundo_vacio=848,
    umbral=0.10,
    diferencia=30,
    seg_min=8,
    seg_libre=1,
    max_frames=None
):
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)

    # 1. Extracción del fondo de referencia
    video.set(cv2.CAP_PROP_POS_MSEC, segundo_vacio * 1000)
    muestras = []
    for _ in range(15):
        ok, f = video.read()
        if ok:
            f = cv2.resize(f, None, fx=escala, fy=escala)
            f = cv2.convertScaleAbs(f, beta=brillo)
            alto, ancho = f.shape[:2]
            y_inicio = int(alto * corte_superior)
            y_fin = int(alto * corte_inferior)
            f = f[y_inicio:y_fin, :]
            muestras.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY))
            
    fondo = np.median(muestras, axis=0).astype(np.uint8)
    fondo = cv2.GaussianBlur(fondo, (5, 5), 0)
    cv2.imwrite('fondo.png', fondo)
    
    # 2. Configurar salida de video
    alto_rec, ancho_rec = fondo.shape[:2]
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho_rec, alto_rec))
    
    # 3. Ajustar zonas al recorte
    zonas_rec = [(x, y - y_inicio, w, h) for x, y, w, h in zonas]
    
    video.set(cv2.CAP_PROP_POS_FRAMES, 0)
    kernel = np.ones((5, 5), np.uint8)
    
    frames_ocupado = [0] * len(zonas_rec)
    frames_libre = [0] * len(zonas_rec)
    contado = [False] * len(zonas_rec)
    contador = [0] * len(zonas_rec)
    historial_eventos = []
    
    print(f"🎬 Procesando {total_frames} fotogramas...")
    pbar = tqdm(total=total_frames, desc="Método Tradicional")
    n_frame = 0
    
    while True:
        ok, img = video.read()
        if not ok or (max_frames and n_frame >= max_frames):
            break
        n_frame += 1
        
        img = cv2.resize(img, None, fx=escala, fy=escala)
        img = cv2.convertScaleAbs(img, beta=brillo)
        img = img[y_inicio:y_fin, :]
        
        imgBN = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
        dif = cv2.absdiff(imgBN, fondo)
        _, imgTH = cv2.threshold(dif, diferencia, 255, cv2.THRESH_BINARY)
        imgMedian = cv2.medianBlur(imgTH, 5)
        imgDil = cv2.dilate(imgMedian, kernel, iterations=2)
        
        for i, (x, y, w, h) in enumerate(zonas_rec):
            zona_binaria = imgDil[y:y+h, x:x+w]
            count = cv2.countNonZero(zona_binaria)
            porcentaje = count / (w * h)
            
            if porcentaje > umbral:
                frames_ocupado[i] += 1
                frames_libre[i] = 0
                if frames_ocupado[i] >= seg_min * fps and not contado[i]:
                    contador[i] += 1
                    contado[i] = True
                    tiempo_seg = n_frame / fps
                    historial_eventos.append((tiempo_seg, i+1, contador[i]))
            else:
                frames_libre[i] += 1
                if frames_libre[i] >= seg_libre * fps:
                    frames_ocupado[i] = 0
                    contado[i] = False
                    
            if contado[i]:
                color = (0, 0, 255)       # Rojo: cliente contado
            elif frames_ocupado[i] > 0:
                color = (0, 255, 255)     # Amarillo: persona en espera
            else:
                color = (0, 255, 0)       # Verde: libre
                
            cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img, f"Z{i+1}: {contador[i]} | {porcentaje*100:.0f}%", (x, y-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                        
        cv2.putText(img, f"CLIENTES: {sum(contador)}", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)
                    
        out.write(img)
        pbar.update(1)
        
    pbar.close()
    video.release()
    out.release()
    
    print("
" + "="*50)
    print("📊 RESULTADOS VISIÓN TRADICIONAL:")
    for i, c in enumerate(contador):
        print(f"   • Zona {i+1}: {c} clientes")
    print(f"   ⭐ TOTAL CLIENTES: {sum(contador)}")
    print(f"   📁 Video guardado: {output_path}")
    print("="*50)
    
    return contador, historial_eventos

# Ejecutar conteo tradicional
# (Si quieres procesar solo una parte de prueba, pasa max_frames=3000)
conteo_trad, eventos_trad = procesar_metodo_tradicional(VIDEO_PATH, 'resultado_tradicional.mp4')


## 📺 Reproducción del Video en el Notebook
El siguiente reproductor convierte el video a formato compatible H.264 para poder verlo directamente en la celda de Jupyter / Google Colab.


In [ ]:
# ==============================================================================
# FUNCIÓN DE REPRODUCCIÓN WEB EN EL NOTEBOOK
# ==============================================================================
def mostrar_video_notebook(video_file, ancho=600):
    web_file = "web_" + video_file
    # Codificar con ffmpeg a H.264 para compatibilidad directa con Chrome/Firefox/Safari
    os.system(f"ffmpeg -y -i {video_file} -vcodec libx264 -crf 23 -pix_fmt yuv420p -acodec aac {web_file} >/dev/null 2>&1")
    target = web_file if os.path.exists(web_file) else video_file
    
    video_bytes = open(target, 'rb').read()
    video_b64 = b64encode(video_bytes).decode()
    
    return HTML(f'''
    <div style="text-align:center;">
        <video width="{ancho}" controls autoplay loop style="border-radius:8px; box-shadow: 0 4px 10px rgba(0,0,0,0.3);">
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
            Tu navegador no soporta reproducción HTML5 directa.
        </video>
    </div>
    ''')

# Visualizar video tradicional
mostrar_video_notebook('resultado_tradicional.mp4', ancho=650)


## 🤖 5. Método 2: Deep Learning con YOLOv11 (Tracking de Personas)

Este método utiliza una red neuronal convolucional moderna (**YOLOv11**) con seguimiento de trayectorias:
- **Modelo**: `yolo11n.pt` (versión Nano, ligera y veloz). Se descarga automáticamente si no está presente.
- **Clase**: `classes=[0]` (exclusivamente personas).
- **Punto de Pies**: Se calcula el punto inferior de la bounding box `(cx, cy) = ((x1+x2)//2, y2)`.
- **Identificador de Cliente**: Cada persona tiene un `ID` único asignado por el rastreador (`ByteTrack`). Cuando sus pies permanecen dentro de alguna zona por más de `SEG_MIN (7 seg)`, se cuenta como cliente atendido sin peligro de conteo doble.


In [ ]:
# ==============================================================================
# 5. EJECUCIÓN MÉTODO YOLOv11
# ==============================================================================
from ultralytics import YOLO

def procesar_metodo_yolo(
    video_path,
    output_path='resultado_yolo.mp4',
    zonas=ZONAS_DEFAULT,
    modelo_nombre='yolo11n.pt',
    seg_min=7,
    salto=3,
    escala=0.7,
    tamano=640,
    max_frames=None
):
    print(f"📦 Cargando modelo YOLO ({modelo_nombre})...")
    modelo = YOLO(modelo_nombre)
    
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)
        
    ancho_frame = int(video.get(cv2.CAP_PROP_FRAME_WIDTH) * escala)
    alto_frame = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT) * escala)
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho_frame, alto_frame))
    
    tiempo_en_zona = {}
    contados = set()
    por_zona = [0] * len(zonas)
    historial_yolo = []
    
    print(f"🎬 Procesando {total_frames} fotogramas con YOLOv11...")
    pbar = tqdm(total=total_frames, desc="Método YOLO")
    n = 0
    
    while True:
        ok, img = video.read()
        if not ok or (max_frames and n >= max_frames):
            break
        n += 1
        
        img = cv2.resize(img, (ancho_frame, alto_frame))
        
        # Procesar con YOLO cada 'salto' fotogramas para máxima fluidez
        if n % salto == 0:
            res = modelo.track(img, persist=True, classes=[0], imgsz=tamano, verbose=False)[0]
            
            if res.boxes.id is not None:
                cajas = res.boxes.xyxy.int().tolist()
                ids = res.boxes.id.int().tolist()
                
                for (x1, y1, x2, y2), id_persona in zip(cajas, ids):
                    cx, cy = (x1 + x2) // 2, y2  # punto de los pies
                    
                    # Verificar en qué zona de compra están sus pies
                    zona_detectada = None
                    for i, (zx, zy, zw, zh) in enumerate(zonas):
                        if zx <= cx <= zx + zw and zy <= cy <= zy + zh:
                            zona_detectada = i
                            break
                            
                    if zona_detectada is not None:
                        tiempo_en_zona[id_persona] = tiempo_en_zona.get(id_persona, 0) + salto
                        if tiempo_en_zona[id_persona] >= seg_min * fps and id_persona not in contados:
                            contados.add(id_persona)
                            por_zona[zona_detectada] += 1
                            historial_yolo.append((n / fps, zona_detectada + 1, id_persona))
                            
                    # Visualización de la persona
                    color = (0, 0, 255) if id_persona in contados else (255, 0, 255)
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, f"ID {id_persona}", (x1, y1 - 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                    cv2.circle(img, (cx, cy), 4, (0, 255, 255), -1)
                    
        # Dibujar zonas de compra
        for i, (zx, zy, zw, zh) in enumerate(zonas):
            cv2.rectangle(img, (zx, zy), (zx + zw, zy + zh), (0, 255, 0), 2)
            cv2.putText(img, f"Z{i+1}: {por_zona[i]}", (zx + 3, zy + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                        
        cv2.putText(img, f"CLIENTES YOLO: {len(contados)}", (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 255), 3)
                    
        out.write(img)
        pbar.update(1)
        
    pbar.close()
    video.release()
    out.release()
    
    print("
" + "="*50)
    print("📊 RESULTADOS YOLOv11:")
    for i, c in enumerate(por_zona):
        print(f"   • Zona {i+1}: {c} clientes")
    print(f"   ⭐ TOTAL CLIENTES ÚNICOS: {len(contados)}")
    print(f"   📁 Video guardado: {output_path}")
    print("="*50)
    
    return por_zona, historial_yolo

# Ejecutar conteo con YOLOv11
conteo_yolo, eventos_yolo = procesar_metodo_yolo(VIDEO_PATH, 'resultado_yolo.mp4')


In [ ]:
# Visualizar video procesado con YOLOv11
mostrar_video_notebook('resultado_yolo.mp4', ancho=650)


## 📊 6. Comparativa de Resultados y Análisis de Rendimiento

A continuación comparamos gráficamente los clientes contabilizados por cada zona y método.


In [ ]:
# ==============================================================================
# 6. GRÁFICA COMPARATIVA
# ==============================================================================
zonas_nombres = [f"Zona {i+1}" for i in range(len(ZONAS_DEFAULT))]
x = np.arange(len(zonas_nombres))
ancho_barra = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - ancho_barra/2, conteo_trad, width=ancho_barra, label='Visión Tradicional (Fondo)', color='#2b5c8f')
plt.bar(x + ancho_barra/2, conteo_yolo, width=ancho_barra, label='Deep Learning (YOLOv11)', color='#e05d44')

plt.xlabel('Zonas de Atención del Quiosco', fontsize=12)
plt.ylabel('Cantidad de Clientes', fontsize=12)
plt.title('Comparativa de Clientes Atendidos por Zona', fontsize=14, fontweight='bold')
plt.xticks(x, zonas_nombres)
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Anotaciones numéricas sobre las barras
for i in range(len(zonas_nombres)):
    plt.text(x[i] - ancho_barra/2, conteo_trad[i] + 0.1, str(conteo_trad[i]), ha='center', fontweight='bold')
    plt.text(x[i] + ancho_barra/2, conteo_yolo[i] + 0.1, str(conteo_yolo[i]), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"📈 Total Clientes - Método Tradicional: {sum(conteo_trad)}")
print(f"📈 Total Clientes - Método YOLOv11:     {sum(conteo_yolo)}")


## 📝 7. Conclusiones y Resumen Técnico

| Característica | Visión Tradicional (Sustracción de Fondo) | Deep Learning (YOLOv11 Tracking) |
| :--- | :--- | :--- |
| **Consumo Computacional** | Muy bajo (corre en cualquier CPU sin tarjeta gráfica) | Moderado / Alto (óptimo con GPU) |
| **Robustez ante Iluminación** | Requiere calibrar umbral y brillo (`BRILLO = 40`) | Muy robusto a sombras y cambios de luz |
| **Identificación Individual** | Basado en áreas de píxeles en la zona | Basado en ID único de persona y tracking de pies |
| **Precisión ante Oclusiones** | Puede agrupar a 2 personas pegadas como 1 sola zona | Separa personas individuales mediante cajas delimitadoras |

### 🚀 Pasos para subir este notebook a Google Colab:
1. Entra a [Google Colab](https://colab.research.google.com).
2. Haz clic en **Subir** y selecciona este archivo `contador_clientes_tienda.ipynb`.
3. Sube `estabilizado.mp4` o `video.mp4` en el panel izquierdo de archivos de Colab (o móntalo desde Google Drive).
4. Ve a `Entorno de ejecución` ➔ `Cambiar tipo de entorno de ejecución` ➔ Activa **T4 GPU** para acelerar YOLO.
5. Ejecuta todas las celdas secuencialmente.
